In [83]:
import pandas as pd

In [84]:
with pd.HDFStore('data/pipeline.h5') as store:
    tables = store.keys()
    df = store['extrapolated_targets']

In [85]:
current_resuts = 'output/LUVit_ct_by_tod_generator-2026-03-19_90-90-90.xlsx'
original_results = 'J:/Projects/LandUseVision/LUV.3/Control_totals/SubJuris_ConTots/LUVit_ct_by_tod_generator_90-90-90_2024-10-18_6575_LUVit_Match.xlsx'

In [86]:
df_current = pd.read_excel(current_resuts, sheet_name='unrolled')
df_original = pd.read_excel(original_results, sheet_name='unrolled')

In [87]:
key_cols = ['subreg_id', 'year']
value_cols = ['total_hhpop', 'total_hh', 'total_emp']

print('Shape check')
print('  current :', df_current.shape)
print('  original:', df_original.shape)
print('  row count match:', len(df_current) == len(df_original))
print()

print('Column check')
print('  current columns :', list(df_current.columns))
print('  original columns:', list(df_original.columns))
print('  columns match:', list(df_current.columns) == list(df_original.columns))
print()

print('Key integrity check')
print('  current duplicate keys :', df_current.duplicated(key_cols).sum())
print('  original duplicate keys:', df_original.duplicated(key_cols).sum())
print()

merged = df_current.merge(
    df_original,
    on=key_cols,
    how='outer',
    suffixes=('_current', '_original'),
    indicator=True,
)

print('Coverage check')
print(merged['_merge'].value_counts(dropna=False).rename_axis('status').to_frame('rows'))
print()

matched = merged.loc[merged['_merge'] == 'both'].copy()
for col in value_cols:
    matched[f'{col}_diff'] = matched[f'{col}_current'] - matched[f'{col}_original']

summary = pd.DataFrame(
    {
        'exact_match_rows': [
            (matched[f'{col}_current'] == matched[f'{col}_original']).sum() for col in value_cols
        ],
        'mismatch_rows': [
            (matched[f'{col}_current'] != matched[f'{col}_original']).sum() for col in value_cols
        ],
        'max_abs_diff': [matched[f'{col}_diff'].abs().max() for col in value_cols],
        'sum_current': [matched[f'{col}_current'].sum() for col in value_cols],
        'sum_original': [matched[f'{col}_original'].sum() for col in value_cols],
        'sum_diff': [matched[f'{col}_diff'].sum() for col in value_cols],
    },
    index=value_cols,
)

print('Value comparison summary')
display(summary)

mismatch_mask = False
for col in value_cols:
    mismatch_mask = mismatch_mask | (matched[f'{col}_current'] != matched[f'{col}_original'])

mismatches = matched.loc[
    mismatch_mask,
    key_cols
    + [f'{col}_current' for col in value_cols]
    + [f'{col}_original' for col in value_cols]
    + [f'{col}_diff' for col in value_cols],
].sort_values(key_cols)

print('Sample mismatched rows')
display(mismatches.head(20))
print('Total mismatched rows:', len(mismatches))
print()

current_year_totals = df_current.groupby('year')[value_cols].sum().sort_index()
original_year_totals = df_original.groupby('year')[value_cols].sum().sort_index()
year_diff = (
    current_year_totals
    .rename(columns=lambda col: f'{col}_current')
    .join(original_year_totals.rename(columns=lambda col: f'{col}_original'))
)
for col in value_cols:
    year_diff[f'{col}_diff'] = year_diff[f'{col}_current'] - year_diff[f'{col}_original']

print('Year-level totals comparison')
display(year_diff)

Shape check
  current : (1477, 5)
  original: (1477, 5)
  row count match: True

Column check
  current columns : ['subreg_id', 'year', 'total_hhpop', 'total_hh', 'total_emp']
  original columns: ['subreg_id', 'year', 'total_hhpop', 'total_hh', 'total_emp']
  columns match: True

Key integrity check
  current duplicate keys : 0
  original duplicate keys: 0

Coverage check
            rows
status          
both        1477
left_only      0
right_only     0

Value comparison summary


,exact_match_rows,mismatch_rows,max_abs_diff,sum_current,sum_original,sum_diff
total_hhpop,16,1461,29740,34300998,34913064,-612066
total_hh,32,1445,24134,13743304,14289457,-546153
total_emp,24,1453,16144,19481874,19860606,-378732


Sample mismatched rows


,subreg_id,year,total_hhpop_current,total_hh_current,total_emp_current,total_hhpop_original,total_hh_original,total_emp_original,total_hhpop_diff,total_hh_diff,total_emp_diff
0,1,2020,82397,30833,42265,87781,32391,42405,-5384,-1558,-140
1,1,2025,84249,31499,42912,89632,33054,43976,-5383,-1555,-1064
2,1,2030,86101,32164,43558,91483,33717,45546,-5382,-1553,-1988
3,1,2035,87954,32830,44204,93334,34380,47117,-5380,-1550,-2913
4,1,2040,89806,33496,44851,95184,35044,48688,-5378,-1548,-3837
5,1,2044,91287,34028,45368,96665,35574,49944,-5378,-1546,-4576
6,1,2050,93510,34827,46144,98886,36370,51829,-5376,-1543,-5685
7,2,2020,208484,88026,109348,209994,90308,112371,-1510,-2282,-3023
8,2,2025,214548,90024,112611,215363,92634,115626,-815,-2610,-3015
9,2,2030,220612,92022,115875,220731,94959,118881,-119,-2937,-3006


Total mismatched rows: 1477

Year-level totals comparison


,total_hhpop_current,total_hh_current,total_emp_current,total_hhpop_original,total_hh_original,total_emp_original,total_hhpop_diff,total_hh_diff,total_emp_diff
year,,,,,,,,,
2020,4053154,1605263,2232229,4209191,1670236,2312316,-156037,-64973,-80087
2025,4338191,1725768,2417628,4471148,1795129,2488970,-132957,-69361,-71342
2030,4623243,1846265,2603021,4733113,1920025,2665623,-109870,-73760,-62602
2035,4908291,1966773,2788421,4995070,2044925,2842275,-86779,-78152,-53854
2040,5193331,2087279,2973822,5257018,2169814,3018930,-63687,-82535,-45108
2044,5421368,2183675,3122139,5466584,2269725,3160255,-45216,-86050,-38116
2050,5763420,2328281,3344614,5780940,2419603,3372237,-17520,-91322,-27623


In [88]:
r_script_inputs = 'data/control_id_working.xlsx'
original_r_script_inputs = 'J:/Projects/LandUseVision/LUV.3/Control_totals/control_id_working_111522.xlsx'

df_current_inputs = pd.read_excel(r_script_inputs)
df_original_inputs = pd.read_excel(original_r_script_inputs)

In [89]:
key_col_inputs = 'control_id'
value_cols_inputs = [
    c for c in df_current_inputs.columns
    if c != key_col_inputs and pd.api.types.is_numeric_dtype(df_current_inputs[c])
]

print('Shape check')
print('  current :', df_current_inputs.shape)
print('  original:', df_original_inputs.shape)
print('  row count match:', len(df_current_inputs) == len(df_original_inputs))
print()

print('Column check')
current_cols  = list(df_current_inputs.columns)
original_cols = list(df_original_inputs.columns)
print('  current columns :', current_cols)
print('  original columns:', original_cols)
print('  columns match:', current_cols == original_cols)
extra_current  = [c for c in current_cols  if c not in original_cols]
extra_original = [c for c in original_cols if c not in current_cols]
if extra_current:
    print('  columns only in current :', extra_current)
if extra_original:
    print('  columns only in original:', extra_original)
print()

print('Key integrity check')
print('  current duplicate keys :', df_current_inputs.duplicated(key_col_inputs).sum())
print('  original duplicate keys:', df_original_inputs.duplicated(key_col_inputs).sum())
print()

merged_inputs = df_current_inputs.merge(
    df_original_inputs,
    on=key_col_inputs,
    how='outer',
    suffixes=('_current', '_original'),
    indicator=True,
)

print('Coverage check')
print(merged_inputs['_merge'].value_counts(dropna=False).rename_axis('status').to_frame('rows'))
print()

shared_value_cols = [
    c for c in value_cols_inputs
    if c in df_original_inputs.columns
]

matched_inputs = merged_inputs.loc[merged_inputs['_merge'] == 'both'].copy()
for col in shared_value_cols:
    matched_inputs[f'{col}_diff'] = matched_inputs[f'{col}_current'] - matched_inputs[f'{col}_original']

summary_inputs = pd.DataFrame(
    {
        'exact_match_rows': [
            (matched_inputs[f'{col}_current'] == matched_inputs[f'{col}_original']).sum()
            for col in shared_value_cols
        ],
        'mismatch_rows': [
            (matched_inputs[f'{col}_current'] != matched_inputs[f'{col}_original']).sum()
            for col in shared_value_cols
        ],
        'max_abs_diff': [matched_inputs[f'{col}_diff'].abs().max() for col in shared_value_cols],
        'sum_current':  [matched_inputs[f'{col}_current'].sum()  for col in shared_value_cols],
        'sum_original': [matched_inputs[f'{col}_original'].sum() for col in shared_value_cols],
        'sum_diff':     [matched_inputs[f'{col}_diff'].sum()     for col in shared_value_cols],
    },
    index=shared_value_cols,
)

print('Value comparison summary')
display(summary_inputs)

mismatch_mask_inputs = False
for col in shared_value_cols:
    mismatch_mask_inputs = mismatch_mask_inputs | (
        matched_inputs[f'{col}_current'] != matched_inputs[f'{col}_original']
    )

mismatches_inputs = matched_inputs.loc[
    mismatch_mask_inputs,
    [key_col_inputs]
    + [f'{col}_current'  for col in shared_value_cols]
    + [f'{col}_original' for col in shared_value_cols]
    + [f'{col}_diff'     for col in shared_value_cols],
].sort_values(key_col_inputs)

print('Sample mismatched rows')
display(mismatches_inputs.head(20))
print('Total mismatched rows:', len(mismatches_inputs))


Shape check
  current : (155, 36)
  original: (155, 28)
  row count match: True

Column check
  current columns : ['control_id', 'target_id', 'name', 'county_id', 'exclude_from_target', 'RGID', 'TotPop20', 'Units20', 'HH20', 'GQ20', 'HHpop20', 'TotEmp20_wCRnoMil', 'Emp_ConRes', 'TotEmpNoMil-ResCon', 'Emp_Military', 'Emp_Total', 'HH44', 'TotPop44', 'GQ44', 'HHpop44', 'TotEmp44_wCRnoMil', 'HH50', 'TotPop50', 'GQ50', 'HHpop50', 'TotEmp50_wCRnoMil', 'Pop18', 'HHpop18', 'Units18', 'HH18', 'GQ18', 'Emp18', 'TotEmpTrg_wCRnoMil', 'TotPopTrg', 'GQpct50', 'PPH50']
  original columns: ['control_id', 'name', 'county_id', 'RGID', 'Pop18', 'HHpop18', 'HH18', 'Emp18', 'TotPop20', 'HHpop20', 'GQpct20', 'HH20', 'PPH20', 'TotPopTrg', 'TotPop44', 'HHPop44', 'HH44', 'TotPop50', 'HHpop50', 'GQpct50', 'HH50', 'PPH50', 'TotEmp20_wCRnoMil', 'Mil20', 'CR20', 'TotEmpTrg_wCRnoMil', 'TotEmp44_wCRnoMil', 'TotEmp50_wCRnoMil']
  columns match: False
  columns only in current : ['target_id', 'exclude_from_target', 'U

,exact_match_rows,mismatch_rows,max_abs_diff,sum_current,sum_original,sum_diff
county_id,155,0,0.000000,7.047000e+03,7.047000e+03,0.000000
RGID,148,7,2.000000,5.770000e+02,5.850000e+02,-8.000000
TotPop20,110,45,939.000000,4.294373e+06,4.294373e+06,0.000000
HH20,112,43,382.000000,1.670235e+06,1.670236e+06,-1.000000
HHpop20,110,45,939.000000,4.209191e+06,4.209191e+06,0.000000
TotEmp20_wCRnoMil,155,0,0.000000,2.312316e+06,2.312316e+06,0.000000
HH44,3,152,3498.949523,2.261558e+06,2.269750e+06,-8191.942722
TotPop44,62,93,9224.000000,5.532930e+06,5.566788e+06,-33858.229980
TotEmp44_wCRnoMil,62,93,2043.000000,3.163246e+06,3.160255e+06,2990.685577
HH50,3,152,4397.186904,2.409393e+06,2.419628e+06,-10235.428402


Sample mismatched rows


,control_id,county_id_current,RGID_current,TotPop20_current,HH20_current,HHpop20_current,TotEmp20_wCRnoMil_current,HH44_current,TotPop44_current,TotEmp44_wCRnoMil_current,...,HHpop50_diff,TotEmp50_wCRnoMil_diff,Pop18_diff,HHpop18_diff,HH18_diff,Emp18_diff,TotEmpTrg_wCRnoMil_diff,TotPopTrg_diff,GQpct50_diff,PPH50_diff
0,1,33,1,151854,60953,150414,155023,92780,228633,230418,...,3878.300250,-0.150920,1719.002057,1475.645055,374.873511,0,18848.679264,19505.168697,-0.702807,0.038514
1,2,33,1,737015,345627,707097,691359,457256,962434,847604,...,73651.925753,-0.193805,-24611.880664,-19746.288600,-4443.595330,0,39061.044956,57264.028650,-3.550381,0.151805
2,3,33,2,77245,27057,76329,49459,37659,102299,69976,...,2381.546998,-0.450155,3628.503495,3407.664500,707.858930,0,5128.839876,6390.856003,-1.020486,0.058965
3,4,33,2,28956,12006,28667,17875,17241,39563,27703,...,764.898498,0.203763,-768.629395,-672.251401,-120.772050,0,2457.163010,2703.328754,-0.828014,0.041154
4,5,33,2,52066,19874,51548,14064,27040,67432,19106,...,1143.030081,-0.309439,-867.637887,-898.403890,-164.805998,0,1260.152448,3751.388621,-0.876275,0.047155
5,6,33,2,100688,36016,99574,32257,46365,123444,53412,...,2915.058486,0.828115,1487.838807,1256.201814,265.175989,0,5289.462492,6142.067862,-1.043562,0.059047
6,7,33,2,39994,16432,39466,30610,19392,44997,37711,...,1269.041217,-0.783024,-245.574903,-392.360903,-353.724471,0,1774.573581,1287.640089,-1.378392,0.065316
7,8,33,2,136555,47055,135030,78269,55785,154317,114750,...,3916.226962,0.185901,2854.113564,3074.427548,-48.760669,0,9120.348721,4643.171819,-1.161665,0.067360
8,9,33,2,92215,38046,91070,57161,49891,115131,82986,...,2078.597934,-0.833031,796.103653,671.003657,544.337432,0,6455.533575,5139.451354,-1.137462,0.056175
9,10,33,2,73256,29693,72834,101631,46482,109128,123765,...,1223.303329,-0.934051,2117.772634,2042.690626,679.053280,0,5532.652760,9128.478653,-0.432276,0.023769


Total mismatched rows: 155


In [90]:
mismatches_inputs.sort_values('HH20_diff')[['control_id','HH20_current','HH20_original','HH20_diff','HH44_current','HH44_original','HH44_diff']]

,control_id,HH20_current,HH20_original,HH20_diff,HH44_current,HH44_original,HH44_diff
76,87,6982,7364,-382,9135,8979.026591,155.973409
150,304,6473,6771,-298,6473,6771.000000,-298.000000
54,64,41139,41306,-167,45823,46572.526100,-749.526100
5,6,36016,36141,-125,46365,46382.252000,-17.252000
46,49,36,139,-103,370,457.247330,-87.247330
...,...,...,...,...,...,...,...
49,53,643,528,115,1096,1068.023350,27.976650
91,124,59238,59103,135,65867,65377.885031,489.114969
147,301,4868,4703,165,4868,4703.000000,165.000000
149,303,2910,2664,246,2910,2664.000000,246.000000


In [91]:
with pd.ExcelWriter('output/mismatches_inputs.xlsx') as writer:
    mismatches_inputs[['control_id','HH20_current','HH20_original','HH20_diff']].to_excel(writer, sheet_name='HH20', index=False)
    mismatches_inputs[['control_id','HH44_current','HH44_original','HH44_diff']].to_excel(writer, sheet_name='HH44', index=False)
    mismatches_inputs[['control_id','TotEmp20_wCRnoMil_current','TotEmp20_wCRnoMil_original','TotEmp20_wCRnoMil_diff']].to_excel(writer, sheet_name='TotEmp20', index=False)
    mismatches_inputs[['control_id','TotEmp44_wCRnoMil_current','TotEmp44_wCRnoMil_original','TotEmp44_wCRnoMil_diff']].to_excel(writer, sheet_name='TotEmp44', index=False)